# Poor Man's Lakehouse — Demo Notebook

This notebook demonstrates the key features of the project. It walks through:

1. **Catalog Browsing** — explore namespaces and tables without a JVM
2. **PyIceberg Client** — schema inspection, snapshot history, Polars scans
3. **Ibis Multi-Engine** — read/write Iceberg tables via DuckDB, PySpark, Polars
4. **Polars Client** — SQL queries with `%%sql` magic
5. **PySpark** — direct Spark access via catalog-specific builders
6. **Sail** — Rust-powered Spark Connect engine (no JVM) with Delta, Iceberg, S3

## Prerequisites

Start the Lakekeeper catalog before running this notebook:

```bash
just up lakekeeper
```

Make sure your `.env` has `CATALOG=lakekeeper`.

For Sail S3 access, set AWS env vars before starting the kernel:

```bash
export AWS_ACCESS_KEY_ID=minioadmin
export AWS_SECRET_ACCESS_KEY=miniopassword
export AWS_ENDPOINT_URL=http://localhost:9000
export AWS_DEFAULT_REGION=eu-central-1
export AWS_ALLOW_HTTP=true
```

---
## 1. Catalog Browsing

The `CatalogBrowser` provides a quick, JVM-free way to explore what's in your catalog.
It auto-resolves the REST catalog URI from your `CATALOG` setting.

In [1]:
from poor_man_lakehouse import CatalogBrowser

browser = CatalogBrowser()
print(f"Catalog: {browser}")
print(f"Namespaces: {browser.list_namespaces()}")

Catalog: CatalogBrowser(catalog='lakekeeper', uri='http://localhost:8181/catalog')
Namespaces: ['default']


In [2]:
# List tables in the default namespace
tables = browser.list_tables("default")
print(f"Tables in 'default': {tables}")

# If tables exist, inspect one
if tables:
    schema = browser.get_table_schema("default", tables[0])
    print(f"\nSchema of '{tables[0]}':")
    for col in schema:
        print(f"  {col['name']:20s} {col['type']:15s} required={col['required']}")

Tables in 'default': []


---
## 2. PyIceberg Client

The `PyIcebergClient` gives full access to Iceberg table metadata — schema evolution,
snapshot history, time travel — plus scanning to Polars or Arrow. No JVM required.

In [3]:
from poor_man_lakehouse import PyIcebergClient

iceberg = PyIcebergClient()
print(iceberg)
print(f"Namespaces: {iceberg.list_namespaces()}")
print(f"Tables in 'default': {iceberg.list_tables('default')}")

PyIcebergClient(catalog='lakekeeper', uri='http://localhost:8181/catalog')
Namespaces: [('default',)]
Tables in 'default': []


In [4]:
# If you have tables, inspect schema and history
tables = iceberg.list_tables("default")
if tables:
    table_name = tables[0][1]  # (namespace, name) tuple
    print(f"\n--- Schema of '{table_name}' ---")
    for col in iceberg.table_schema("default", table_name):
        print(f"  {col['name']:20s} {col['type']}")

    print("\n--- Snapshot History ---")
    for snap in iceberg.snapshot_history("default", table_name):
        print(
            f"  ID: {snap['snapshot_id']}  ts: {snap['timestamp_ms']}  ops: {snap['summary'].get('operation', 'N/A')}"
        )

In [5]:
# Scan a table to Polars LazyFrame (lazy evaluation)
if tables:
    table_name = tables[0][1]
    lf = iceberg.scan_to_polars("default", table_name)
    print(f"LazyFrame schema: {lf.collect_schema()}")
    display(lf.head(5).collect())

---
## 3. Ibis Multi-Engine Connection

The `IbisConnection` lets you access the same Iceberg tables through PySpark, Polars,
and DuckDB — all via a unified Ibis interface. DuckDB 1.5+ also supports **writing**
to Iceberg tables.

In [6]:
from poor_man_lakehouse import IbisConnection

conn = IbisConnection()
print(f"Catalog: {conn._catalog_name}")
print(f"Endpoint: {conn._lakekeeper_endpoint}")

Catalog: lakekeeper
Endpoint: http://localhost:8181/catalog


In [7]:
# Create a table and write data via DuckDB
conn.create_table("default", "demo_users", "id INTEGER, name VARCHAR, age INTEGER")
print("Table created!")

# Insert rows
conn.write_table(
    "default",
    "demo_users",
    "duckdb",
    query="SELECT 1 AS id, 'Alice' AS name, 30 AS age UNION ALL SELECT 2, 'Bob', 25 UNION ALL SELECT 3, 'Charlie', 35",
)
print("Data written!")

2026-03-29 20:02:44.307 | INFO     | poor_man_lakehouse.ibis_connector.builder:create_table:360 - Created table lakekeeper.default.demo_users
2026-03-29 20:02:44.367 | INFO     | poor_man_lakehouse.ibis_connector.builder:write_table:343 - Wrote to lakekeeper.default.demo_users (mode=append) via DuckDB


Table created!
Data written!


In [8]:
# Read with DuckDB
result = conn.sql("SELECT * FROM lakekeeper.default.demo_users", "duckdb")
print("--- DuckDB read ---")
result.execute()

--- DuckDB read ---


,id,name,age
0,1,Alice,30
1,2,Bob,25
2,3,Charlie,35


In [9]:
# Read the same table with Polars (via PyIceberg bridge)
polars_table = conn.read_table("default", "demo_users", "polars")
print("--- Polars read ---")
polars_table.execute()

--- Polars read ---


,id,name,age
0,1,Alice,30
1,2,Bob,25
2,3,Charlie,35


In [10]:
# Clean up
conn.close()

---
## 4. Polars Client with Lakekeeper Backend

The `PolarsClient` provides a SQL interface using Polars. With `backend="lakekeeper"`,
it uses PyIceberg under the hood to scan tables.

In [11]:
from poor_man_lakehouse import PolarsClient

client = PolarsClient(backend="lakekeeper")
print(f"Catalogs: {client.list_catalogs()}")
print(f"Namespaces: {client.list_namespaces('lakekeeper')}")
print(f"Tables: {client.list_tables('lakekeeper', 'default')}")

Catalogs: ['lakekeeper']
Namespaces: ['default']
Tables: ['demo_users']


In [12]:
# Scan a table directly
lf = client.scan_table("lakekeeper", "default", "demo_users")
display(lf.collect())

id,name,age
i32,str,i32
1,"""Alice""",30
2,"""Bob""",25
3,"""Charlie""",35


In [13]:
# Load the %%sql magic for Jupyter
from poor_man_lakehouse import load_sql_magic

load_sql_magic(client)

✓ SQL magic loaded. Use %%sql to query Unity Catalog tables.
  Example:
    %%sql
    SELECT * FROM unity.default.test_table


In [14]:
%%sql
SELECT * FROM lakekeeper.default.demo_users
WHERE age > 25
ORDER BY name

id,name,age
i32,str,i32
1,"""Alice""",30
3,"""Charlie""",35


---
## 5. PySpark Direct Access

For full Spark ecosystem access, use the catalog-specific builders.

In [15]:
from poor_man_lakehouse import CatalogType, get_spark_builder

builder = get_spark_builder(CatalogType.LAKEKEEPER)
spark = builder.get_spark_session()
print(f"Spark version: {spark.version}")
print(f"Catalog: {spark.catalog.currentCatalog()}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/29 20:02:46 WARN Utils: Your hostname, MacBookPro.homenet.telecomitalia.it, resolves to a loopback address: 127.0.0.1; using 192.168.1.81 instead (on interface en0)
26/03/29 20:02:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/graziano/apache-spark/4.0.1/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/graziano/.ivy2.5.2/cache
The jars for the packages stored in: /Users/graziano/.ivy2.5.2/jars
io.delta#delta-spark_2.13 added as a dependency
org.apache.iceberg#iceberg-spark-runtime-4.0_2.13 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
org.postgresql#postgresql added as a dependency
org.projectnessie.nessie-integrations#nessie-spark-extensions-3.5_2.13 added as a dependency
io.unitycatalog#unitycat

Spark version: 4.0.1
Catalog: lakekeeper


In [16]:
# Query the table we created earlier
spark.sql("SELECT * FROM lakekeeper.default.demo_users").show()

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 30|
|  2|    Bob| 25|
|  3|Charlie| 35|
+---+-------+---+



In [17]:
# Insert more data via Spark
spark.sql("""
    INSERT INTO lakekeeper.default.demo_users
    VALUES (4, 'Diana', 28), (5, 'Eve', 32)
""")
spark.sql("SELECT * FROM lakekeeper.default.demo_users ORDER BY id").show()

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 30|
|  2|    Bob| 25|
|  3|Charlie| 35|
|  4|  Diana| 28|
|  5|    Eve| 32|
+---+-------+---+



In [18]:
spark.stop()

---
## Summary

| Connector | JVM Required | Read | Write | Best For |
|-----------|:---:|:---:|:---:|----------|
| `CatalogBrowser` | No | Metadata only | No | Quick exploration |
| `PyIcebergClient` | No | Polars, Arrow | No | Schema, snapshots, scans |
| `IbisConnection` | PySpark only | All 3 engines | DuckDB | Multi-engine comparison |
| `PolarsClient` | No | Polars | No | SQL + `%%sql` magic |
| `SparkBuilder` | Yes | PySpark | PySpark | Full Spark ecosystem |
| `Sail` | No (Rust) | PySpark API | Delta, Iceberg, Parquet | JVM-free Spark alternative |
| `DremioConnection` | No | Arrow Flight | No | Query federation |

---
## 6. Sail — Rust-Powered Spark Connect Engine (No JVM)

[Sail](https://docs.lakesail.com/sail/latest/) is a Rust-based compute engine that implements
the Spark Connect protocol. It provides a PySpark-compatible API **without a JVM**, with native
support for Parquet, Delta Lake, Iceberg (local), and S3/MinIO.

Sail reads S3 credentials from environment variables (`AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`,
`AWS_ENDPOINT_URL`, `AWS_ALLOW_HTTP`), not from Spark config.

> **Note:** Sail does not support REST catalogs (Lakekeeper/Nessie) yet. It works with
> local file-based table formats and S3 storage directly.

In [ ]:
from pysail.spark import SparkConnectServer
from pyspark.sql import SparkSession

# Start Sail server (Rust engine, no JVM needed)
server = SparkConnectServer()
server.start()
addr = server.listening_address
print(f"Sail server listening on {addr[0]}:{addr[1]}")

# Connect PySpark via Spark Connect protocol
sail_spark = SparkSession.builder.remote(f"sc://{addr[0]}:{addr[1]}").getOrCreate()
print(f"Spark version (via Sail): {sail_spark.version}")

In [ ]:
# Basic DataFrame operations — same PySpark API, Rust engine under the hood
df = sail_spark.createDataFrame(
    [(1, "Alice", 30), (2, "Bob", 25), (3, "Charlie", 35), (4, "Diana", 28)],
    ["id", "name", "age"],
)
df.show()

# SQL analytics
df.createOrReplaceTempView("sail_users")
sail_spark.sql("SELECT name, age, RANK() OVER (ORDER BY age DESC) as rank FROM sail_users").show()

In [ ]:
# Delta Lake on S3/MinIO (requires AWS_* env vars set before starting the kernel)
# If running from terminal: export AWS_ACCESS_KEY_ID=minioadmin AWS_SECRET_ACCESS_KEY=miniopassword ...
try:
    df.write.format("delta").mode("overwrite").save("s3://warehouse/sail_demo/delta")
    sail_spark.read.format("delta").load("s3://warehouse/sail_demo/delta").show()
    print("Sail + Delta on S3: OK")
except Exception as e:
    print(f"Sail + Delta on S3: {e}")
    print("(Set AWS_* env vars before starting the kernel for S3 access)")

In [ ]:
# Local Iceberg (file-based, no catalog server needed)
import tempfile

tmp = tempfile.mkdtemp()
df.write.format("iceberg").mode("overwrite").save(f"file://{tmp}/sail_iceberg/")
sail_spark.read.format("iceberg").load(f"file://{tmp}/sail_iceberg/").show()
print("Sail + Iceberg (local): OK")

In [ ]:
sail_spark.stop()
server.stop()
print("Sail server stopped")